# Sprint 5 — Modèles nanoGPT par Catégorie

Ce notebook entraîne **un modèle par catégorie/langue** :
- `model_fr_tech.pt`
- `model_fr_food.pt`
- `model_fr_luxe.pt`
- `model_ar_tech.pt`
- `model_ar_food.pt`
- `model_ar_luxe.pt`

Ces modèles sont chargés **dynamiquement** par `nomgen_core.py` selon le secteur demandé par l'API.
Si un modèle n'existe pas, le service utilise le modèle générique comme fallback.

In [1]:
import os
import sys
import json
import torch
from pathlib import Path

# Chemins
PROJECT_ROOT = Path(os.getcwd()).parent
BACKEND_PATH = PROJECT_ROOT / 'backend'
DATA_DIR     = PROJECT_ROOT / 'data'
WEIGHTS_DIR  = BACKEND_PATH / 'app' / 'weights'

if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(0, str(BACKEND_PATH))

from app.models.nanogpt import NanoGPT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Environnement prêt  | Périphérique : {device.upper()}')
print(f'DATA_DIR    : {DATA_DIR}')
print(f'WEIGHTS_DIR : {WEIGHTS_DIR}')

Environnement prêt  | Périphérique : CPU
DATA_DIR    : c:\Users\Lenovo\Documents\ProjetVersion1\data
WEIGHTS_DIR : c:\Users\Lenovo\Documents\ProjetVersion1\backend\app\weights


In [2]:
# Charger les vocabulaires existants
with open(WEIGHTS_DIR / 'vocab_fr.json', encoding='utf-8') as f:
    vocab_fr_data = json.load(f)

with open(WEIGHTS_DIR / 'vocab_ar.json', encoding='utf-8') as f:
    vocab_ar_data = json.load(f)

stoi_fr = vocab_fr_data['stoi']
stoi_ar = vocab_ar_data['stoi']

itos_fr = {int(v): k for k, v in stoi_fr.items()}
itos_ar = {int(v): k for k, v in stoi_ar.items()}

vocab_size_fr = len(stoi_fr)
vocab_size_ar = len(stoi_ar)

print(f'Vocab FR : {vocab_size_fr} caractères')
print(f'Vocab AR : {vocab_size_ar} caractères')

Vocab FR : 33 caractères
Vocab AR : 39 caractères


In [5]:
# ── Fonctions utilitaires ─────────────────────────────────────────────────────

def load_category_data(langue: str, categorie: str, type_nom: str = None) -> str:
    """
    Charge le texte d'entraînement pour une catégorie donnée.
    Si type_nom est None, charge marque + societe.
    """
    texts = []
    types = [type_nom] if type_nom else ['marque', 'societe']
    
    for typ in types:
        path = DATA_DIR / langue / typ / f'{categorie}.txt'
        if path.exists():
            with open(path, encoding='utf-8') as f:
                content = f.read().strip()
                if content:
                    texts.append(content)
            print(f'   Chargé : {path.relative_to(PROJECT_ROOT)} ({len(content.split(chr(10)))} noms)')
        else:
            print(f'    Absent : {path.relative_to(PROJECT_ROOT)}')
    
    return '\n'.join(texts) + '\n' if texts else ''


def prepare_tensors(text: str, stoi: dict):
    """Encode le texte en tenseurs PyTorch, en ignorant les caractères inconnus."""
    # Filtrer les caractères inconnus du vocabulaire
    filtered = [c for c in text if c in stoi]
    if len(filtered) < 50:
        return None, None
    
    data   = torch.tensor([stoi[c] for c in filtered], dtype=torch.long)
    n      = int(0.9 * len(data))
    return data[:n], data[n:]


def get_batch(train_data, val_data, split, block_size=24, batch_size=32):
    data_split = train_data if split == 'train' else val_data
    if len(data_split) <= block_size:
        return None, None
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x  = torch.stack([data_split[i:i+block_size] for i in ix])
    y  = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)


def train_model(train_data, val_data, vocab_size, model_name,
                max_iters=2000, eval_interval=500, lr=1e-3):
    """
    Entraîne un nanoGPT sur les données fournies.
    Retourne le modèle entraîné.
    """
    model = NanoGPT(vocab_size=vocab_size, n_embed=64, n_head=4,
                    n_layer=4, block_size=24).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    print(f'\n Entraînement : {model_name} | {sum(p.numel() for p in model.parameters()):,} params')
    
    for it in range(max_iters):
        xb, yb = get_batch(train_data, val_data, 'train')
        if xb is None:
            print('    Données insuffisantes  entraînement annulé.')
            return None
        
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        
        if it % eval_interval == 0 or it == max_iters - 1:
            model.eval()
            with torch.no_grad():
                xv, yv = get_batch(train_data, val_data, 'val')
                if xv is not None:
                    _, lv = model(xv, yv)
                    print(f'  Iter {it:4d} | Train: {loss.item():.4f} | Val: {lv.item():.4f}')
            model.train()
    
    return model


def save_model(model, model_name):
    """Sauvegarde le modèle dans backend/app/weights/."""
    if model is None:
        print(f'  Modèle non sauvegardé ({model_name} — données insuffisantes)')
        return
    path = WEIGHTS_DIR / f'{model_name}.pt'
    torch.save(model.to('cpu').state_dict(), path)
    print(f'  Sauvegardé : {path.name}')

print('Fonctions prêtes ')

Fonctions prêtes 


In [6]:
# ── Modèle FR Tech ───────────────────────────────────────────────────────────
print('=== Modèle FR Tech ===')
text = load_category_data('fr', 'tech')

if text.strip():
    train_d, val_d = prepare_tensors(text, stoi_fr)
    if train_d is not None:
        model = train_model(train_d, val_d, vocab_size_fr, 'model_fr_tech')
        save_model(model, 'model_fr_tech')
    else:
        print('    Texte insuffisant pour FR Tech')
else:
    print('    Aucun fichier data/fr/*/tech.txt trouvé')

=== Modèle FR Tech ===
   Chargé : data\fr\marque\tech.txt (495 noms)
   Chargé : data\fr\societe\tech.txt (257 noms)

 Entraînement : model_fr_tech | 204,800 params
  Iter    0 | Train: 3.5227 | Val: 3.3544
  Iter  500 | Train: 2.0235 | Val: 2.0977
  Iter 1000 | Train: 1.5566 | Val: 2.0277
  Iter 1500 | Train: 1.4444 | Val: 2.0930
  Iter 1999 | Train: 1.0053 | Val: 2.3711
  Sauvegardé : model_fr_tech.pt


In [7]:
# ── Modèle FR Food ───────────────────────────────────────────────────────────
print('=== Modèle FR Food ===')
text = load_category_data('fr', 'food')

if text.strip():
    train_d, val_d = prepare_tensors(text, stoi_fr)
    if train_d is not None:
        model = train_model(train_d, val_d, vocab_size_fr, 'model_fr_food')
        save_model(model, 'model_fr_food')
    else:
        print('    Texte insuffisant pour FR Food')
else:
    print('   Aucun fichier data/fr/*/food.txt trouvé')

=== Modèle FR Food ===
   Chargé : data\fr\marque\food.txt (270 noms)
   Chargé : data\fr\societe\food.txt (260 noms)

 Entraînement : model_fr_food | 204,800 params
  Iter    0 | Train: 3.5035 | Val: 3.2485
  Iter  500 | Train: 2.0544 | Val: 1.9455
  Iter 1000 | Train: 1.5436 | Val: 2.0289
  Iter 1500 | Train: 1.0737 | Val: 2.4417
  Iter 1999 | Train: 0.7523 | Val: 2.7837
  Sauvegardé : model_fr_food.pt


In [8]:
# ── Modèle FR Luxe ───────────────────────────────────────────────────────────
print('=== Modèle FR Luxe ===')
text = load_category_data('fr', 'luxe')

if text.strip():
    train_d, val_d = prepare_tensors(text, stoi_fr)
    if train_d is not None:
        model = train_model(train_d, val_d, vocab_size_fr, 'model_fr_luxe')
        save_model(model, 'model_fr_luxe')
    else:
        print('    Texte insuffisant pour FR Luxe')
else:
    print('    Aucun fichier data/fr/*/luxe.txt trouvé')

=== Modèle FR Luxe ===
   Chargé : data\fr\marque\luxe.txt (278 noms)
   Chargé : data\fr\societe\luxe.txt (212 noms)

 Entraînement : model_fr_luxe | 204,800 params
  Iter    0 | Train: 3.4868 | Val: 3.3119
  Iter  500 | Train: 1.9822 | Val: 2.4925
  Iter 1000 | Train: 1.1607 | Val: 2.3186
  Iter 1500 | Train: 0.8533 | Val: 1.8376
  Iter 1999 | Train: 0.6771 | Val: 1.7246
  Sauvegardé : model_fr_luxe.pt


In [9]:
# ── Modèle AR General (enrichi) ──────────────────────────────────────────────
# On combine toutes les catégories arabes pour avoir assez de données
print('=== Modèle AR General (toutes catégories) ===')

all_ar_texts = []
for cat in ['general', 'tech', 'food', 'luxe']:
    t = load_category_data('ar', cat)
    if t.strip():
        all_ar_texts.append(t)

text_ar = '\n'.join(all_ar_texts)

if text_ar.strip():
    train_d, val_d = prepare_tensors(text_ar, stoi_ar)
    if train_d is not None:
        model = train_model(train_d, val_d, vocab_size_ar, 'model_ar',
                           max_iters=2000)
        save_model(model, 'model_ar')
    else:
        print('    Texte insuffisant pour AR')
else:
    print('    Aucun fichier arabe trouvé')

=== Modèle AR General (toutes catégories) ===
   Chargé : data\ar\marque\general.txt (621 noms)
   Chargé : data\ar\societe\general.txt (127 noms)
   Chargé : data\ar\marque\tech.txt (234 noms)
   Chargé : data\ar\societe\tech.txt (414 noms)
   Chargé : data\ar\marque\food.txt (200 noms)
   Chargé : data\ar\societe\food.txt (252 noms)
   Chargé : data\ar\marque\luxe.txt (209 noms)
   Chargé : data\ar\societe\luxe.txt (329 noms)

 Entraînement : model_ar | 205,568 params
  Iter    0 | Train: 3.6994 | Val: 3.4734
  Iter  500 | Train: 2.2425 | Val: 2.5977
  Iter 1000 | Train: 2.2773 | Val: 2.5090
  Iter 1500 | Train: 2.1346 | Val: 2.4692
  Iter 1999 | Train: 1.9068 | Val: 2.6689
  Sauvegardé : model_ar.pt


In [10]:
# ── Modèle AR Tech ───────────────────────────────────────────────────────────
print('=== Modèle AR Tech ===')
text = load_category_data('ar', 'tech')

if len(text.strip().split('\n')) >= 30:  # minimum 30 noms
    train_d, val_d = prepare_tensors(text, stoi_ar)
    if train_d is not None:
        model = train_model(train_d, val_d, vocab_size_ar, 'model_ar_tech')
        save_model(model, 'model_ar_tech')
else:
    print(' Pas assez de noms arabes tech (< 30). Utiliser le modèle générique.')

=== Modèle AR Tech ===
   Chargé : data\ar\marque\tech.txt (234 noms)
   Chargé : data\ar\societe\tech.txt (414 noms)

 Entraînement : model_ar_tech | 205,568 params
  Iter    0 | Train: 3.6886 | Val: 3.5245
  Iter  500 | Train: 2.0865 | Val: 3.0659
  Iter 1000 | Train: 1.5341 | Val: 3.6246
  Iter 1500 | Train: 1.0530 | Val: 4.3897
  Iter 1999 | Train: 0.8252 | Val: 4.8443
  Sauvegardé : model_ar_tech.pt


In [15]:
# ── Validation : tester l'inférence de chaque modèle généré ─────────────────
import torch.nn.functional as F

def test_inference(model_path: Path, stoi: dict, itos: dict, n_test: int = 5):
    """Génère quelques noms avec le modèle sauvegardé."""
    if not model_path.exists():
        print(f'   {model_path.name} introuvable')
        return
    
    vocab_size = len(stoi)
    model = NanoGPT(vocab_size=vocab_size, n_embed=64, n_head=4, n_layer=4, block_size=24)
    model.load_state_dict(torch.load(str(model_path), map_location='cpu'))
    model.eval()
    
    noms = []
    pad_id = stoi.get('.', 0)
    
    for _ in range(n_test * 3):
        ctx = torch.zeros((1, 24), dtype=torch.long)
        chars = []
        for _ in range(15):
            logits, _ = model(ctx)
            logits = logits[:, -1, :] / 0.8
            v, _ = torch.topk(logits, 20)
            logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            ix = torch.multinomial(probs, 1).item()
            if ix == pad_id:
                break
            ch = itos.get(ix, '')
            if ch.startswith('#'):
                break
            chars.append(ch)
            new = torch.roll(ctx, -1, dims=1).clone()
            new[0, -1] = ix
            ctx = new
        nm = ''.join(chars).strip()
        if 3 <= len(nm) <= 12:
            noms.append(nm)
        if len(noms) >= n_test:
            break
    
    print(f'  {model_path.name} → {noms[:n_test]}')


print('\n=== Validation des modèles générés ===')
for model_file in sorted(WEIGHTS_DIR.glob('model_*.pt')):
    stoi = stoi_fr if '_ar' not in model_file.stem else stoi_ar
    itos = itos_fr if '_ar' not in model_file.stem else itos_ar
    test_inference(model_file, stoi, itos)

print('\n Sprint 5 terminé ! Modèles par catégorie exportés vers backend/app/weights/')


=== Validation des modèles générés ===
  model_ar.pt → ['موكودوليك', 'موجاكولوهو', 'سيبايكاكوبر', 'بوواجاهرة', 'مومونا']
  model_ar_tech.pt → ['أوتوريدا', 'أورابط', 'لوجيتلابر', 'أنجاثر', 'شاربيمجةوج']
  model_fr.pt → ['eerrrginio', 'arebonepler', 'bogormegecta', 'cadulia', 'rastelllle']
  model_fr_food.pt → ['eliérésia', 'avoraîchine', 'astivoraft', 'olvianaina', 'lpicardell']
  model_fr_luxe.pt → ['acharel', 'arioiana', 'arenes', 'arioana', 'aurio']
  model_fr_tech.pt → ['gotalia', 'gronaota', 'megaraore', 'rogicab', 'natanoha']

 Sprint 5 terminé ! Modèles par catégorie exportés vers backend/app/weights/
